In [3]:
# Colab用：/content/ball_list.json を読み込み、卓球台4隅(画像座標)を使って
# 「バウンド(bounce)」と「打球(hit)」を推定します。
#
# 追加したこと：
# - 卓球台の4隅からホモグラフィHを作り、ボール座標を台平面の正規化座標(u,v)へ射影
# - bounce判定に「台ポリゴン内にいること」を必須条件として追加（誤検出が大きく減ります）
#
# 今回の対応：
# - hit判定のしきい値を下げる（デフォルト値を緩める＋呼び出し側も緩める）
# - hitとbounceが「かぶった」場合は hit を優先（重なりbounceを除去）
#
# 前提：120fps、yは下向き増加、(0,0)は欠損扱い、prodは信頼度(文字列/数値どちらでもOK)

import json, os, sys
import math
import numpy as np
import matplotlib.pyplot as plt
import cv2
from typing import Tuple

# OpenCV（Colabは通常入ってます）
import cv2

# --- もしSavitzky–Golayを使いたいなら（推奨） ---
# !pip -q install scipy

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Add project root directory to sys.path
project_folder_name = 'datapingpong-vision-lab/02_bound-detection'  # <--- update if needed
project_path = os.path.join('/content/drive/MyDrive', project_folder_name)
sys.path.append(project_path)

FPS = 120.0
JSON_PATH = "data/DJI_0056_001.MP4.json"

# ============================
# 卓球台4隅（画像座標）
# ============================
# 解釈： (近側左, 遠側左, 遠側右, 近側右)
table_corners_img = np.array([
    [180.0, 240.0],  # near-left  (bottom-left)
    [200.0, 160.0],  # far-left   (top-left)
    [440.0, 160.0],  # far-right  (top-right)
    [460.0, 240.0],  # near-right (bottom-right)
], dtype=np.float32)

# 台平面の正規化座標へ対応付け（u:左右 0..1, v:奥行 0..1）
# far側(top)を v=0、near側(bottom)を v=1 とする
table_corners_uv = np.array([
    [0.0, 1.0],  # near-left
    [0.0, 0.0],  # far-left
    [1.0, 0.0],  # far-right
    [1.0, 1.0],  # near-right
], dtype=np.float32)

# ホモグラフィ（画像 -> 台(u,v)）
H_img_to_uv = cv2.getPerspectiveTransform(table_corners_img, table_corners_uv)

# 台ポリゴン（画像座標）: point-in-polygon用（OpenCVはintが楽）
table_poly_img = table_corners_img.reshape((-1, 1, 2)).astype(np.int32)

# ============================
# ユーティリティ
# ============================
def wrap_angle(rad: np.ndarray) -> np.ndarray:
    return (rad + np.pi) % (2.0 * np.pi) - np.pi

def interp_nan_1d(arr: np.ndarray) -> np.ndarray:
    idx = np.arange(arr.shape[0], dtype=float)
    m = ~np.isnan(arr)
    if m.sum() < 5:
        return arr
    return np.interp(idx, idx[m], arr[m])

def moving_average(arr: np.ndarray, win: int) -> np.ndarray:
    if win <= 1:
        return arr
    win = int(win)
    pad = win // 2
    x = np.pad(arr, (pad, pad), mode="edge")
    k = np.ones(win, dtype=float) / float(win)
    return np.convolve(x, k, mode="valid")

def smooth_savgol_or_ma(arr: np.ndarray, window: int = 9, poly: int = 2) -> np.ndarray:
    window = int(window)
    if window % 2 == 0:
        window += 1
    window = max(window, poly + 3 + (poly + 3) % 2)
    try:
        from scipy.signal import savgol_filter
        return savgol_filter(arr, window_length=window, polyorder=poly, mode="interp")
    except Exception:
        return moving_average(arr, win=window)

def central_diff(arr: np.ndarray) -> np.ndarray:
    n = arr.shape[0]
    d = np.zeros(n, dtype=float)
    if n < 2:
        return d
    d[1:-1] = (arr[2:] - arr[:-2]) / 2.0
    d[0] = arr[1] - arr[0]
    d[-1] = arr[-1] - arr[-2]
    return d

def peak_pick(idxs, score: np.ndarray, merge_gap: int = 2):
    if not idxs:
        return []
    idxs = sorted(set(int(i) for i in idxs))
    groups = [[idxs[0]]]
    for t in idxs[1:]:
        if t - groups[-1][-1] <= merge_gap:
            groups[-1].append(t)
        else:
            groups.append([t])
    return [max(g, key=lambda t: float(score[t])) for g in groups]

def robust_quantile(x: np.ndarray, q: float) -> float:
    x = x[np.isfinite(x)]
    if x.size == 0:
        return float("inf")
    return float(np.quantile(x, q))

def load_ball_list(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    xs, ys, ps = [], [], []
    for obj in data:
        xs.append(float(obj.get("x", 0)))
        ys.append(float(obj.get("y", 0)))
        p = obj.get("prod", 0.0)
        try:
            ps.append(float(p))
        except Exception:
            ps.append(0.0)
    return np.array(xs, dtype=float), np.array(ys, dtype=float), np.array(ps, dtype=float)

def interpolate_short_gaps(arr: np.ndarray, max_gap: int) -> np.ndarray:
    n = arr.shape[0]
    out = arr.copy()
    isn = np.isnan(out)
    if not isn.any():
        return out
    i = 0
    while i < n:
        if not isn[i]:
            i += 1
            continue
        j = i
        while j < n and isn[j]:
            j += 1
        gap_len = j - i
        if gap_len <= max_gap:
            left = i - 1
            right = j
            if left >= 0 and right < n and np.isfinite(out[left]) and np.isfinite(out[right]):
                out[i:j] = np.linspace(out[left], out[right], gap_len + 2)[1:-1]
        i = j
    return out

def preprocess(x, y, p,
               p_min=0.65,
               max_interp_gap=10,
               smooth_window=9,
               smooth_poly=2):
    valid = (p >= p_min) & ~((x == 0.0) & (y == 0.0))
    x2 = x.copy(); y2 = y.copy()
    x2[~valid] = np.nan
    y2[~valid] = np.nan

    x2 = interpolate_short_gaps(x2, max_interp_gap)
    y2 = interpolate_short_gaps(y2, max_interp_gap)

    # 連続性を優先した全体補間（必要なら後で区間分割に切替可能）
    x2 = interp_nan_1d(x2)
    y2 = interp_nan_1d(y2)

    x2 = smooth_savgol_or_ma(x2, window=smooth_window, poly=smooth_poly)
    y2 = smooth_savgol_or_ma(y2, window=smooth_window, poly=smooth_poly)
    return x2, y2

def compute_kinematics(x, y):
    vx = central_diff(x)
    vy = central_diff(y)
    ax = central_diff(vx)
    ay = central_diff(vy)

    speed = np.hypot(vx, vy)
    acc = np.hypot(ax, ay)

    theta = np.arctan2(vy, vx)
    dtheta = np.zeros_like(theta)
    dtheta[1:-1] = np.abs(wrap_angle(theta[2:] - theta[:-2]))

    dv = np.zeros_like(speed)
    dv[1:-1] = np.abs(speed[2:] - speed[:-2]) / 2.0
    return vx, vy, ax, ay, speed, acc, dtheta, dv

def project_img_to_uv(xs: np.ndarray, ys: np.ndarray, H: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """(x,y) -> (u,v) (正規化台座標)"""
    pts = np.stack([xs, ys], axis=1).astype(np.float32).reshape((-1, 1, 2))
    uv = cv2.perspectiveTransform(pts, H).reshape((-1, 2))
    return uv[:, 0].astype(float), uv[:, 1].astype(float)

def inside_table_polygon(xs: np.ndarray, ys: np.ndarray, poly_img_int: np.ndarray) -> np.ndarray:
    """画像上で台ポリゴン内なら True"""
    inside = np.zeros(xs.shape[0], dtype=bool)
    for i, (x, y) in enumerate(zip(xs, ys)):
        # pointPolygonTest: >0 inside, 0 on edge, <0 outside
        inside[i] = (cv2.pointPolygonTest(poly_img_int, (float(x), float(y)), False) >= 0)
    return inside

# ============================
# 検出（ホモグラフィ情報を盛り込み）
# ============================
def detect_events(
    x_raw, y_raw, p_raw,
    fps=FPS,

    # 前処理
    p_min=0.65,
    max_interp_gap=10,
    smooth_window=5,
    smooth_poly=2,

    # しきい値（分位点）
    bounce_a_q=0.94,

    # ---- hit (SIDE VIEW: x方向の進行方向反転のみ) ----
    hit_vx_min_q=0.1,   # |vx|が弱い反転(ノイズ)を除外：多すぎ→上げる / 抜け→下げる
    hit_ax_q=0.1,       # |ax|が小さい反転(なだらか)を除外：多すぎ→上げる / 抜け→下げる
    hit_bounce_penalty=0.4,  # バウンド近傍hitのスコア減衰(0.2〜0.7)

    # 近傍処理
    overlap_frames=3,         # ← 「かぶり」の判定幅（±このフレーム）
    merge_gap_frames=2,

    # bounceの追加条件
    require_local_max_y=True,     # yは下向き増加なので「局所最大」がバウンドらしい
    require_inside_table=True,    # 台ポリゴン内を必須にする
):
    x_s, y_s = preprocess(
        x_raw, y_raw, p_raw,
        p_min=p_min,
        max_interp_gap=max_interp_gap,
        smooth_window=smooth_window,
        smooth_poly=smooth_poly
    )

    vx, vy, ax, ay, speed, acc, dtheta, dv = compute_kinematics(x_s, y_s)

    # 台平面(u,v)へ射影（確認やフィルタに利用）
    u_s, v_s = project_img_to_uv(x_s, y_s, H_img_to_uv)

    # 台ポリゴン内判定（画像上）
    inside_tbl = inside_table_polygon(x_s, y_s, table_poly_img)

    # 閾値
    a_thr_bounce = robust_quantile(acc, bounce_a_q)

    def is_local_max_y(y, t):
        return (y[t] >= y[t-1]) and (y[t] >= y[t+1])

    n = len(x_s)

    # ---- Bounce ----
    bounce_cand = []
    for t in range(2, n - 2):
        # y下向き増加：落下 vy>0 → 反転して vy<0
        sign_flip = (vy[t - 1] > 0.0) and (vy[t + 1] < 0.0)
        if not sign_flip:
            continue
        if acc[t] < a_thr_bounce:
            continue
        if require_inside_table and (not inside_tbl[t]):
            continue
        if require_local_max_y and (not is_local_max_y(y_s, t)):
            continue
        bounce_cand.append(t)

    bounce_frames = peak_pick(bounce_cand, score=acc, merge_gap=merge_gap_frames)

    # ---- Hit (SIDE VIEW: x方向の進行方向反転のみ) ----
    vx_h = vx
    ax_h = ax

    # 閾値（分位点で自動調整）
    vx_min = robust_quantile(np.abs(vx_h), hit_vx_min_q)
    ax_thr = robust_quantile(np.abs(ax_h), hit_ax_q)

    def near_bounce(t):
        return any(abs(t - b) <= overlap_frames for b in bounce_frames)

    hit_cand = []
    hit_score = np.zeros(n, dtype=float)

    for t in range(2, n - 2):
        # x方向の進行方向が反転（符号反転） OR 急減速（ブロック/軽打を拾う）
        vx_flip = (vx_h[t-1] * vx_h[t+1] < 0)

        # 急減速：打球で速度が落ちるケース（符号は変わらないことが多い）
        # 例：t-1 の |vx| に対して t+1 が ratio 未満
        vx_brake_ratio = 0.45  # 0.35〜0.60で調整。小さいほど厳しい。
        vx_brake = (abs(vx_h[t+1]) < abs(vx_h[t-1]) * vx_brake_ratio)

        if not (vx_flip or vx_brake):
            continue

        # 弱い反転（|vx|が小さいところでの符号ブレ）を除外
        if min(abs(vx_h[t-1]), abs(vx_h[t+1])) < vx_min:
            continue

        # なだらかな反転（カーブ）を除外：急な反転だけ残す
        if abs(ax_h[t]) < ax_thr:
            continue

        # スコア：反転の強さ（単純）
        s = abs(ax_h[t]) * (abs(vx_h[t-1]) + abs(vx_h[t+1]))

        # バウンド近傍は「禁止」ではなく減点（hit優先のまま整合）
        if near_bounce(t):
            s *= float(hit_bounce_penalty)

        hit_cand.append(t)
        hit_score[t] = s

    hit_frames = peak_pick(hit_cand, score=hit_score, merge_gap=merge_gap_frames)


    # ---- Overlap resolution: hit優先（かぶりbounceを除去）----
    if len(hit_frames) > 0 and len(bounce_frames) > 0:
        bounce_frames = [
            b for b in bounce_frames
            if all(abs(b - h) > overlap_frames for h in hit_frames)
        ]

    debug = {
        "x_smooth": x_s, "y_smooth": y_s,
        "u_smooth": u_s, "v_smooth": v_s,
        "inside_table": inside_tbl,

        "vx": vx, "vy": vy, "ax": ax, "ay": ay,
        "speed": speed, "acc": acc, "dtheta": dtheta, "dv": dv,

        "a_thr_bounce": a_thr_bounce,
        "overlap_frames": overlap_frames,
        "vx_min_hit": vx_min,
        "ax_thr_hit": ax_thr,
        "hit_vx_min_q": hit_vx_min_q,
        "hit_ax_q": hit_ax_q,
        "hit_bounce_penalty": hit_bounce_penalty,
    }
    return bounce_frames, hit_frames, debug

# ============================
# 実行
# ============================
x_raw, y_raw, p_raw = load_ball_list(os.path.join(project_path, JSON_PATH))

bounces, hits, dbg = detect_events(
    x_raw, y_raw, p_raw,
    fps=FPS,
    p_min=0.65,

    # bounceは現状維持〜微調整
    bounce_a_q=0.80,
    overlap_frames=3,     # かぶり判定幅（hit優先）
    require_local_max_y=True,
    require_inside_table=True,
)

print("Bounce frames:", bounces)
print("Hit frames   :", hits)
print("\nBounce times (s):", [round(t / FPS, 4) for t in bounces])
print("Hit times (s)   :", [round(t / FPS, 4) for t in hits])

print("\nThresholds:")
print("  a_thr_bounce =", dbg["a_thr_bounce"])
print("  overlap_frames =", dbg["overlap_frames"])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Bounce frames: [63, 104, 156, 838, 899, 1219, 1268, 1344, 1430, 1519, 1591, 1687, 1729, 1901, 2000, 2055, 2173, 2402, 2451, 2520, 2592, 2679, 2779, 2873, 3399, 3456, 3499, 3753, 3802, 3851, 3933, 4038, 4098, 4152, 4200, 4366, 4420, 4885, 4988, 5065, 5139, 5224, 5284, 5329, 5362, 5889, 6018, 6063, 6132, 6168, 6176, 6189, 7335, 7383, 7408, 7552, 7638, 7684, 7799, 7867, 7916, 8070, 8848, 8966, 9039, 9743, 9879, 9922, 10001, 10097, 10189, 10271, 10532, 10573, 10612, 10952, 11016, 11489, 11890, 11937, 12036, 12154, 12264, 12506, 12551, 12708, 13020, 13071, 13134, 13168, 13449, 13527, 13545, 13746, 13871, 14010, 14091, 14182, 14194, 14342, 14477, 14521, 14582, 14677, 14738, 14750, 14995, 15021, 15056, 15103, 15444, 15650, 15763, 15816, 15934, 15976, 16076, 16128, 16242, 16305, 16647, 16702, 16754, 18195, 18239, 18298, 18352, 18410, 18504, 18570, 18659, 18713, 18794

In [4]:

# ============================
# 可視化：x(t), y(t), u(t), v(t) とイベント線
# ============================
tsec = np.arange(len(dbg["x_smooth"])) / FPS

def draw_vlines(frames, style):
    for fr in frames:
        plt.axvline(fr / FPS, linestyle=style, linewidth=1)

# x(t)
plt.figure(figsize=(12,4))
plt.plot(tsec, dbg["x_smooth"])
draw_vlines(bounces, "--")
draw_vlines(hits, ":")
plt.title("x(t) (bounce=--, hit=:)")
plt.xlabel("time (s)")
plt.ylabel("x (px)")
plt.grid(True)
plt.show()

# y(t)
plt.figure(figsize=(12,4))
plt.plot(tsec, dbg["y_smooth"])
draw_vlines(bounces, "--")
draw_vlines(hits, ":")
plt.title("y(t) (downwards +) (bounce=--, hit=:)")
plt.xlabel("time (s)")
plt.ylabel("y (px)")
plt.grid(True)
plt.show()

# u(t), v(t) (台平面正規化座標)
plt.figure(figsize=(12,4))
plt.plot(tsec, dbg["u_smooth"])
draw_vlines(bounces, "--")
draw_vlines(hits, ":")
plt.title("u(t) (table-normalized, left=0 right=1)")
plt.xlabel("time (s)")
plt.ylabel("u")
plt.grid(True)
plt.show()

plt.figure(figsize=(12,4))
plt.plot(tsec, dbg["v_smooth"])
draw_vlines(bounces, "--")
draw_vlines(hits, ":")
plt.title("v(t) (table-normalized, far=0 near=1)")
plt.xlabel("time (s)")
plt.ylabel("v")
plt.grid(True)
plt.show()

# inside_table(t)
plt.figure(figsize=(12,2.8))
plt.plot(tsec, dbg["inside_table"].astype(int))
draw_vlines(bounces, "--")
draw_vlines(hits, ":")
plt.title("inside_table(t) (1=in table polygon)")
plt.xlabel("time (s)")
plt.ylabel("inside")
plt.ylim(-0.1, 1.1)
plt.grid(True)
plt.show()

# acc(t)
plt.figure(figsize=(12,4))
plt.plot(tsec, dbg["acc"])
plt.axhline(dbg["a_thr_bounce"], linestyle="--", linewidth=1)
draw_vlines(bounces, "--")
draw_vlines(hits, ":")
plt.title("|a|(t) (horizontal: bounce_thr=--, hit_thr=:)")
plt.xlabel("time (s)")
plt.ylabel("|a|")
plt.grid(True)
plt.show()

# ============================
# Bounce frames の位置を表示
# ============================
x_s = dbg["x_smooth"]
y_s = dbg["y_smooth"]
u_s = dbg["u_smooth"]
v_s = dbg["v_smooth"]
inside = dbg["inside_table"]

print("Bounce frame positions (smoothed):")
print("frame\t time(s)\t x\t y\t u\t v\t inside_table")
for fr in bounces:
    if 0 <= fr < len(x_s):
        print(
            f"{fr}\t {fr/FPS:.4f}\t"
            f"{x_s[fr]:.2f}\t {y_s[fr]:.2f}\t"
            f"{u_s[fr]:.4f}\t {v_s[fr]:.4f}\t"
            f"{int(inside[fr])}"
        )

# ============================
# Hit frames の位置も表示（ついで）
# ============================
print("\nHit frame positions (smoothed):")
print("frame\t time(s)\t x\t y\t u\t v\t inside_table")
for fr in hits:
    if 0 <= fr < len(x_s):
        print(
            f"{fr}\t {fr/FPS:.4f}\t"
            f"{x_s[fr]:.2f}\t {y_s[fr]:.2f}\t"
            f"{u_s[fr]:.4f}\t {v_s[fr]:.4f}\t"
            f"{int(inside[fr])}"
        )


Output hidden; open in https://colab.research.google.com to view.

In [5]:
all_events = []

# Process bounce events
for fr in bounces:
    if 0 <= fr < len(dbg["x_smooth"]):
        event_data = {
            "event": "bounce",
            "frame": fr + 9,
            "time_s": round(fr / FPS, 4),
            "x": round(dbg["x_smooth"][fr], 2),
            "y": round(dbg["y_smooth"][fr], 2),
            "u": round(dbg["u_smooth"][fr], 4),
            "v": round(dbg["v_smooth"][fr], 4)
        }
        all_events.append(event_data)

# Process hit events
for fr in hits:
    if 0 <= fr < len(dbg["x_smooth"]):
        event_data = {
            "event": "hit",
            "frame": fr + 9,
            "time_s": round(fr / FPS, 4),
            "x": round(dbg["x_smooth"][fr], 2),
            "y": round(dbg["y_smooth"][fr], 2)
        }
        all_events.append(event_data)

# Sort events chronologically by frame number
all_events.sort(key=lambda x: x['frame'])

# Define the output file path
output_file_name = "annotated_events.json"
output_file_path = os.path.join(project_path, output_file_name)

# Save the data to a JSON file
with open(output_file_path, "w", encoding="utf-8") as f:
    json.dump(all_events, f, indent=4)

print(f"Annotated events saved to: {output_file_path}")

Annotated events saved to: /content/drive/MyDrive/datapingpong-vision-lab/02_bound-detection/annotated_events.json


In [6]:
import os
import cv2
import json

VIDEO_PATH = os.path.join(project_path, "../01_ball-tracking/data/DJI_0056_001.MP4")
OUTPUT_PATH = os.path.join(project_path, "output_bounce_check.mp4")
EVENT_JSON_PATH = os.path.join(project_path, "annotated_events.json")

with open(EVENT_JSON_PATH, "r") as f:
    events = json.load(f)

# --- ここから生成（JSON -> bounce_frames / ball_positions） ---

# bounceイベントだけ抽出
bounce_events = [e for e in events if e.get("event") == "bounce" and "frame" in e]

# frame番号リスト（重複除去 + ソート）
bounce_frames = sorted({int(e["frame"]) for e in bounce_events})

hit_events = [e for e in events if e.get("event") == "hit" and "frame" in e]
hit_frames = sorted({int(e["frame"]) for e in hit_events})

hit_positions = {}
for e in hit_events:
    fr = int(e["frame"])
    hit_positions[fr] = (float(e["x"]), float(e["y"]))

# frame -> (x, y) を作る
# 優先順位: x,y があればそれを使う。なければ u,v から復元する（0-1正規化）
ball_positions = {}
for e in bounce_events:
    fr = int(e["frame"])
    ball_positions[fr] = (float(e["x"]), float(e["y"]))

# --- ここまで生成 ---
COLOR_BOUNCE = (0, 0, 255)   # 赤
COLOR_HIT    = (0, 255, 0)   # 緑

MARK_DURATION = 60  # フレーム数
RADIUS = 10
# 学習・検証用の解像度
OUT_WIDTH = 640
OUT_HEIGHT = 320

cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps == 0:
    fps = 30

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    OUTPUT_PATH,
    fourcc,
    fps,
    (OUT_WIDTH, OUT_HEIGHT)
)
frame_idx = 0

# 「今どのバウンドを表示中か」を管理
active_marks = []  # [(end_frame, (x,y), color)]

bounce_set = set(bounce_frames)

# ログを減らしたい場合はここを True に
VERBOSE = True

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (OUT_WIDTH, OUT_HEIGHT), interpolation=cv2.INTER_AREA)

    # 新しいバウンドが来たら登録
    if frame_idx in bounce_set:
        pos = ball_positions.get(frame_idx)
        if pos is not None:
            x, y = pos
            active_marks.append((frame_idx + MARK_DURATION, (int(x), int(y)), COLOR_BOUNCE))
            if VERBOSE:
                print("append:", active_marks[-1])

    # --- hit ---
    if frame_idx in hit_frames:
        pos = hit_positions.get(frame_idx)
        if pos is not None:
            x, y = pos
            active_marks.append(
                (frame_idx + MARK_DURATION, (int(x), int(y)), COLOR_HIT)
            )

    # 有効期限切れを削除
    active_marks = [
        (end_f, pos, col) for end_f, pos, col in active_marks
        if frame_idx <= end_f
    ]

    # マーキング描画
    for _, (x, y), col in active_marks:
        cv2.circle(frame, (x, y), RADIUS, col, 2)  # thickness=2 → 枠

    # フレーム番号表示
    cv2.putText(
        frame,
        f"Frame: {frame_idx}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (255, 255, 255),
        2
    )

    writer.write(frame)
    frame_idx += 1

    if VERBOSE and frame_idx % 300 == 0:
        print("processed:", frame_idx)

cap.release()
writer.release()

print("saved:", OUTPUT_PATH)


append: (132, (228, 169), (0, 0, 255))
append: (173, (431, 192), (0, 0, 255))
append: (225, (278, 193), (0, 0, 255))
processed: 300
processed: 600
append: (907, (379, 196), (0, 0, 255))
processed: 900
append: (968, (252, 170), (0, 0, 255))
processed: 1200
append: (1288, (271, 166), (0, 0, 255))
append: (1337, (385, 176), (0, 0, 255))
append: (1413, (301, 178), (0, 0, 255))
append: (1499, (367, 173), (0, 0, 255))
processed: 1500
append: (1588, (288, 175), (0, 0, 255))
append: (1660, (388, 206), (0, 0, 255))
append: (1756, (274, 180), (0, 0, 255))
append: (1798, (211, 160), (0, 0, 255))
processed: 1800
append: (1970, (208, 179), (0, 0, 255))
append: (2069, (270, 174), (0, 0, 255))
append: (2124, (379, 202), (0, 0, 255))
processed: 2100
append: (2242, (447, 200), (0, 0, 255))
processed: 2400
append: (2471, (415, 221), (0, 0, 255))
append: (2520, (278, 188), (0, 0, 255))
append: (2589, (428, 206), (0, 0, 255))
append: (2661, (279, 187), (0, 0, 255))
append: (2748, (421, 197), (0, 0, 255))
